# C2-linear-models — Practice p20

**Type:** constrained coding · **Difficulty:** core · **Concepts:** linear-regression-estimator-derivation

## Part A — derivation (4 points, separately scored)

Starting from
$$L(\beta)=\frac{1}{n}\lVert X\beta-y\rVert_2^2,$$
derive
$$\nabla_{\beta}L=\frac{2}{n}X^{\mathsf T}(X\beta-y)$$
and then the normal equations
$$(X^{\mathsf T}X)\beta=X^{\mathsf T}y.$$
Explain why full column rank of `X` makes the Gram matrix `X.T @ X` positive definite and invertible, and why that makes the normal-equation solution unique. Finally, connect the mathematical quantities to the implementation names `G = X.T @ X` and `c = X.T @ y` used in Part B.

### Student derivation response

<!-- Write your derivation and explanation below. -->



### Solution

Let $X\in\mathbb{R}^{n\times p}$, $\beta\in\mathbb{R}^p$, and $y\in\mathbb{R}^n$, and define the residual $r=X\beta-y$. Expanding the squared norm gives
$$
L(\beta)=\frac{1}{n}\sum_{i=1}^n\left(\sum_{j=1}^p X_{ij}\beta_j-y_i\right)^2.
$$
For each coordinate $k\in\{1,\ldots,p\}$, the chain rule gives
$$
\frac{\partial L}{\partial\beta_k}
=\frac{2}{n}\sum_{i=1}^n X_{ik}
\left(\sum_{j=1}^p X_{ij}\beta_j-y_i\right)
=\frac{2}{n}(X^{\mathsf T}r)_k.
$$
Stacking these $p$ partial derivatives and substituting $r=X\beta-y$ yields
$$
\nabla_{\beta}L=\frac{2}{n}X^{\mathsf T}(X\beta-y).
$$
At a minimizer, the first-order condition is
$$
0=\frac{2}{n}X^{\mathsf T}(X\beta-y).
$$
Multiplying by $n/2$ and rearranging gives the normal equations
$$
(X^{\mathsf T}X)\beta=X^{\mathsf T}y.
$$
If $X$ has full column rank, then its null space contains only $0$. Therefore, for every nonzero $v\in\mathbb{R}^p$,
$$
v^{\mathsf T}(X^{\mathsf T}X)v=(Xv)^{\mathsf T}(Xv)=\lVert Xv\rVert_2^2>0.
$$
Thus $X^{\mathsf T}X$ is positive definite, so it is invertible. The normal equations consequently have exactly one solution. In Part B, `G = X.T @ X` is the matrix on the left, `c = X.T @ y` is the right-hand side, and `np.linalg.solve(G, c)` returns that unique coefficient vector $\beta$ without explicitly forming an inverse.

### Part A rubric

1. **Gradient (1 point):** correctly derives the stated gradient from the squared-residual loss, with dimensions or intermediate residual reasoning shown.
2. **Normal equations (1 point):** correctly sets the first-order condition and obtains the stated normal system.
3. **Uniqueness (1 point):** correctly explains the full-column-rank implication for positive definiteness, invertibility, and uniqueness.
4. **Code connection (1 point):** explicitly identifies `G` and `c` with the two sides of the normal equations and states what `solve(G, c)` returns.

## Part B — constrained implementation

Implement \`ols_full_rank(X, y)\`; an intercept is already a column of
$X$.

Accept finite numeric \`X (n, p)\` and \`y (n,)\` only when $n\ge p$,
row counts match, and \`np.linalg.matrix_rank(X) == p\`.
Raise \`ValueError\` before solving for malformed, non-finite, or
rank-deficient input. Do not mutate inputs.

Form $G=X^TX$ and $c=X^Ty$, then make exactly one
\`np.linalg.solve(G, c)\` call. If floating-point formation makes $G$
numerically singular and that solve raises \`np.linalg.LinAlgError\`,
convert it to \`ValueError\` after the one attempted call. This is the
normal-equation method's numerical limitation; grading success fixtures
have a solvable computed Gram matrix.

On success, return finite float \`beta (p,)\`. RTOL is used only for
the coefficient and residual comparisons with \`ATOL = 1e-10\`,
\`RTOL = 1e-10\`; it does not
apply to the normal-system or orthogonality-to-zero checks. Those use
exactly
\`ATOL + BACKWARD_SAFETY * eps * max(1, effective_condition) * max(1, problem_scale)\`,
where \`eps = np.finfo(float).eps\` and \`BACKWARD_SAFETY = 64.0\`.
Here \`effective_condition = min(cond(G), 1 / eps)\`, with a non-finite
\`cond(G)\` treated as \`1 / eps\`. For the normal-system check,
\`problem_scale = ||G||_inf ||beta||_inf + ||c||_inf\`; for the
orthogonality check, it is \`||X.T||_inf ||X beta - y||_inf\`.

The function and its helpers have a constrained grading namespace.
They may load the exact \`np\` global, the provided \`ATOL\`, \`RTOL\`,
\`FLOAT_EPS\`, and \`BACKWARD_SAFETY\` constants, main-module helper
functions, and only these builtins: \`ValueError\`, \`float\`, \`int\`,
\`bool\`, \`len\`, \`all\`, and \`any\`. Permitted NumPy/data
attribute names are \`array\`, \`asarray\`, \`isfinite\`,
\`issubdtype\`, \`number\`, \`floating\`, \`integer\`, \`linalg\`,
\`matrix_rank\`, \`solve\`, \`LinAlgError\`, \`ndim\`, \`shape\`,
\`size\`, \`dtype\`, \`astype\`, \`all\`, \`copy\`, and \`T\`.
Helpers are recursively audited under the same allowlist. Custom
classes, callable objects, partials, bound methods, extra modules,
unknown attributes, imports, and runtime namespace/reflection routes
such as \`globals\`, \`locals\`, \`vars\`, \`eval\`, \`exec\`,
\`compile\`, \`__import__\`, \`getattr\`, \`setattr\`, and
\`delattr\` are not permitted. This is a grading bytecode namespace
contract, not a security sandbox.

**Banned inside the function and any helper it calls (zero points):**
\`np.linalg.inv\`, \`np.linalg.pinv\`, \`np.linalg.lstsq\`,
\`getattr\`, \`__dict__\`, any spelling of \`sklearn\` or
\`statsmodels\`, loops, comprehensions, recursion, or saved aliases to
forbidden routines.

The immutable checker instruments required/forbidden calls, inspects
nested code and referenced globals/defaults/closures, uses scaled
secondary fixtures, and verifies that output propagates from the solve
return rather than a dummy call.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 1e-10
FLOAT_EPS = np.finfo(float).eps
BACKWARD_SAFETY = 64.0


def ols_full_rank(X, y):
    X_array = np.asarray(X)
    y_array = np.asarray(y)
    X_numeric = (np.issubdtype(X_array.dtype, np.floating) or
                 np.issubdtype(X_array.dtype, np.integer))
    y_numeric = (np.issubdtype(y_array.dtype, np.floating) or
                 np.issubdtype(y_array.dtype, np.integer))
    if (X_array.ndim != 2 or y_array.ndim != 1 or
            X_array.size == 0 or y_array.size == 0 or
            X_array.shape[0] != y_array.shape[0] or
            X_array.shape[0] < X_array.shape[1] or
            not X_numeric or not y_numeric or
            not np.isfinite(X_array).all() or
            not np.isfinite(y_array).all()):
        raise ValueError("invalid OLS input")
    X_float = X_array.astype(float, copy=True)
    y_float = y_array.astype(float, copy=True)
    if np.linalg.matrix_rank(X_float) != X_float.shape[1]:
        raise ValueError("X must have full column rank")
    gram = X_float.T @ X_float
    rhs = X_float.T @ y_float
    try:
        beta = np.linalg.solve(gram, rhs)
    except np.linalg.LinAlgError as error:
        raise ValueError("computed Gram matrix is singular") from error
    beta_float = np.asarray(beta).astype(float, copy=True)
    if (beta_float.ndim != 1 or
            beta_float.shape[0] != X_float.shape[1] or
            not np.isfinite(beta_float).all()):
        raise ValueError("solve returned an invalid coefficient vector")
    return beta_float

## Immutable contract check — do not edit

The checker independently verifies accepted, scaled, rejected, and
computed-Gram-singular cases. It checks exact solve arguments, patches
every forbidden NumPy route, recursively audits code and captured
aliases, and uses a controlled solve return to prove data flow.

In [ ]:
import dis
import builtins
import inspect
import types

_ORIGINAL_SOLVE_P20 = np.linalg.solve
_ORIGINAL_INV_P20 = np.linalg.inv
_ORIGINAL_PINV_P20 = np.linalg.pinv
_ORIGINAL_LSTSQ_P20 = np.linalg.lstsq
_FORBIDDEN_FUNCS_P20 = (
    _ORIGINAL_INV_P20, _ORIGINAL_PINV_P20, _ORIGINAL_LSTSQ_P20,
)
assert FLOAT_EPS == np.finfo(float).eps
assert BACKWARD_SAFETY == 64.0
_APPROVED_HELPER_MODULE_P20 = "__main__"
_SAFE_CONSTANT_TYPES_P20 = (
    type(None), bool, int, float, complex, str, bytes,
)
_ALLOWED_BUILTIN_NAMES_P20 = {
    "ValueError", "float", "int", "bool", "len", "all", "any",
}
_ALLOWED_GLOBAL_CONSTANTS_P20 = {
    "ATOL", "RTOL", "FLOAT_EPS", "BACKWARD_SAFETY",
}
_ALLOWED_ATTR_NAMES_P20 = {
    "array", "asarray", "isfinite", "issubdtype", "number",
    "floating", "integer", "linalg", "matrix_rank", "solve",
    "LinAlgError", "ndim", "shape", "size", "dtype", "astype",
    "all", "copy", "T",
}
_BANNED_NAMESPACE_NAMES_P20 = {
    "globals", "locals", "vars", "eval", "exec", "compile",
    "__import__", "getattr", "setattr", "delattr", "dir",
    "hasattr",
}
_SAFE_BUILTINS_P20 = tuple(
    vars(builtins)[name] for name in _ALLOWED_BUILTIN_NAMES_P20
)


def _audit_value_p20(value, seen, pending_functions):
    marker = id(value)
    if marker in seen:
        return
    seen.add(marker)
    assert all(value is not item for item in _FORBIDDEN_FUNCS_P20)

    if value is np:
        return
    if isinstance(value, types.FunctionType):
        assert value.__module__ == _APPROVED_HELPER_MODULE_P20
        pending_functions.append(value)
    elif isinstance(value, dict):
        for key, item in value.items():
            _audit_value_p20(key, seen, pending_functions)
            _audit_value_p20(item, seen, pending_functions)
    elif isinstance(value, (tuple, list, set, frozenset)):
        for item in value:
            _audit_value_p20(item, seen, pending_functions)
    elif type(value) in _SAFE_CONSTANT_TYPES_P20:
        return
    else:
        assert any(value is item for item in _SAFE_BUILTINS_P20)


_pending_functions_p20 = [ols_full_rank]
_seen_functions_p20 = set()
_codes_p20 = []
while _pending_functions_p20:
    _function_p20 = _pending_functions_p20.pop()
    if id(_function_p20) in _seen_functions_p20:
        continue
    assert _function_p20.__module__ == _APPROVED_HELPER_MODULE_P20
    _seen_functions_p20.add(id(_function_p20))
    _defaults_p20 = (
        tuple(_function_p20.__defaults__ or ())
        + tuple((_function_p20.__kwdefaults__ or {}).values())
    )
    for _value_p20 in _defaults_p20:
        _audit_value_p20(_value_p20, set(), _pending_functions_p20)
    for _cell_p20 in (_function_p20.__closure__ or ()):
        _audit_value_p20(_cell_p20.cell_contents, set(), _pending_functions_p20)

    _code_pending_p20 = [_function_p20.__code__]
    while _code_pending_p20:
        _code_p20 = _code_pending_p20.pop()
        _codes_p20.append(_code_p20)
        _code_pending_p20.extend(
            item for item in _code_p20.co_consts
            if isinstance(item, types.CodeType)
        )
        _names_p20 = {name.lower() for name in _code_p20.co_names}
        assert not (_names_p20 & {
            "inv", "pinv", "lstsq", "getattr", "__dict__",
            "sklearn", "statsmodels",
        })
        assert _function_p20.__name__ not in _code_p20.co_names
        _ops_p20 = {item.opname for item in dis.get_instructions(_code_p20)}
        assert not (_ops_p20 & {
            "FOR_ITER", "IMPORT_NAME", "LOAD_BUILD_CLASS",
            "STORE_ATTR", "DELETE_ATTR",
        })
        assert not any(name.startswith("JUMP_BACKWARD") for name in _ops_p20)
        for _instruction_p20 in dis.get_instructions(_code_p20):
            _name_p20 = _instruction_p20.argval
            if _instruction_p20.opname in {"LOAD_ATTR", "LOAD_METHOD"}:
                assert _name_p20 in _ALLOWED_ATTR_NAMES_P20
            elif _instruction_p20.opname in {"LOAD_GLOBAL", "LOAD_NAME"}:
                assert _name_p20 not in _BANNED_NAMESPACE_NAMES_P20
                assert not (
                    _name_p20.startswith("__")
                    or _name_p20.endswith("__")
                )
                if _name_p20 == "np":
                    assert _function_p20.__globals__.get(_name_p20) is np
                elif _name_p20 in _ALLOWED_GLOBAL_CONSTANTS_P20:
                    _constant_p20 = _function_p20.__globals__[_name_p20]
                    assert type(_constant_p20) in _SAFE_CONSTANT_TYPES_P20
                elif _name_p20 in _ALLOWED_BUILTIN_NAMES_P20:
                    if _name_p20 in _function_p20.__globals__:
                        assert (
                            _function_p20.__globals__[_name_p20]
                            is vars(builtins)[_name_p20]
                        )
                else:
                    _helper_p20 = _function_p20.__globals__[_name_p20]
                    assert isinstance(_helper_p20, types.FunctionType)
                    _audit_value_p20(
                        _helper_p20, set(), _pending_functions_p20,
                    )

try:
    _source_p20 = inspect.getsource(ols_full_rank).lower()
except (OSError, TypeError):
    _source_p20 = ""
assert all(token not in _source_p20 for token in (
    "np.linalg.inv(", "np.linalg.pinv(", "np.linalg.lstsq(",
    "getattr(", "__dict__", "sklearn", "statsmodels",
))

_X1_p20 = np.array([
    [1.0, -2.0, 0.0],
    [1.0, 0.0, 1.0],
    [1.0, 1.0, -1.0],
    [1.0, 3.0, 2.0],
    [1.0, 4.0, 0.0],
])
_y1_p20 = np.array([-0.2, 0.8, 2.2, 6.4, 6.3])
_X2_p20 = np.array([
    [1.0, -3.0],
    [1.0, -1.0],
    [1.0, 2.0],
    [1.0, 4.0],
    [1.0, 6.0],
    [1.0, 9.0],
])
_y2_p20 = np.array([-4.0, -0.5, 4.2, 7.1, 10.5, 14.8])
_X3_p20 = np.array([
    [1.0, 1.0],
    [1.0, 1.0 + 3e-7],
    [1.0, 1.0 - 3e-7],
    [1.0, 1.0 + 6e-7],
])
_y3_p20 = np.arange(4.0)
_accepted_p20 = (
    (_X1_p20, _y1_p20, True),
    (_X2_p20, _y2_p20, True),
    (_X2_p20 * 1e8, _y2_p20 * 1e8, True),
    (_X3_p20, _y3_p20, False),
)


def _invoke_p20(X, y, solve_replacement):
    forbidden_calls = []

    def forbid(name):
        def blocked(*args, **kwargs):
            forbidden_calls.append(name)
            raise AssertionError(f"forbidden route called: {name}")
        return blocked

    np.linalg.solve = solve_replacement
    np.linalg.inv = forbid("inv")
    np.linalg.pinv = forbid("pinv")
    np.linalg.lstsq = forbid("lstsq")
    try:
        result = ols_full_rank(X, y)
    finally:
        np.linalg.solve = _ORIGINAL_SOLVE_P20
        np.linalg.inv = _ORIGINAL_INV_P20
        np.linalg.pinv = _ORIGINAL_PINV_P20
        np.linalg.lstsq = _ORIGINAL_LSTSQ_P20
    assert forbidden_calls == []
    return result


for _X_p20, _y_p20, _compare_lstsq_p20 in _accepted_p20:
    _G_p20 = _X_p20.T @ _X_p20
    _c_p20 = _X_p20.T @ _y_p20
    _expected_p20 = _ORIGINAL_SOLVE_P20(_G_p20, _c_p20)
    if _compare_lstsq_p20:
        _lstsq_p20 = _ORIGINAL_LSTSQ_P20(_X_p20, _y_p20, rcond=None)[0]
        assert np.allclose(
            _expected_p20, _lstsq_p20, atol=ATOL, rtol=RTOL,
        )
    _solve_calls_p20 = []

    def counted_solve(a, b):
        _solve_calls_p20.append(
            (np.array(a, copy=True), np.array(b, copy=True))
        )
        return _ORIGINAL_SOLVE_P20(a, b)

    _X_before_p20 = _X_p20.copy()
    _y_before_p20 = _y_p20.copy()
    _beta_p20 = _invoke_p20(_X_p20, _y_p20, counted_solve)
    assert len(_solve_calls_p20) == 1
    assert np.array_equal(_solve_calls_p20[0][0], _G_p20)
    assert np.array_equal(_solve_calls_p20[0][1], _c_p20)
    assert isinstance(_beta_p20, np.ndarray)
    assert _beta_p20.shape == (_X_p20.shape[1],)
    assert np.issubdtype(_beta_p20.dtype, np.floating)
    assert np.isfinite(_beta_p20).all()
    assert np.array_equal(_X_p20, _X_before_p20)
    assert np.array_equal(_y_p20, _y_before_p20)
    assert np.allclose(
        _beta_p20, _expected_p20, atol=ATOL, rtol=RTOL,
    )
    _resid_p20 = _X_p20 @ _beta_p20 - _y_p20
    _expected_resid_p20 = _X_p20 @ _expected_p20 - _y_p20
    assert np.allclose(
        _resid_p20, _expected_resid_p20, atol=ATOL, rtol=RTOL,
    )
    _normal_gap_p20 = np.linalg.norm(
        _G_p20 @ _beta_p20 - _c_p20, ord=np.inf,
    )
    _normal_scale_p20 = (
        np.linalg.norm(_G_p20, ord=np.inf)
        * np.linalg.norm(_beta_p20, ord=np.inf)
        + np.linalg.norm(_c_p20, ord=np.inf)
    )
    _raw_cond_p20 = np.linalg.cond(_G_p20)
    _effective_cond_p20 = min(
        _raw_cond_p20 if np.isfinite(_raw_cond_p20) else 1 / FLOAT_EPS,
        1 / FLOAT_EPS,
    )
    _normal_bound_p20 = (
        ATOL
        + BACKWARD_SAFETY
        * FLOAT_EPS
        * max(1.0, _effective_cond_p20)
        * max(1.0, _normal_scale_p20)
    )
    assert np.isfinite(_normal_bound_p20)
    assert _normal_gap_p20 <= _normal_bound_p20
    _orth_gap_p20 = np.linalg.norm(
        _X_p20.T @ _resid_p20, ord=np.inf,
    )
    _orth_scale_p20 = (
        np.linalg.norm(_X_p20.T, ord=np.inf)
        * np.linalg.norm(_resid_p20, ord=np.inf)
    )
    _orth_bound_p20 = (
        ATOL
        + BACKWARD_SAFETY
        * FLOAT_EPS
        * max(1.0, _effective_cond_p20)
        * max(1.0, _orth_scale_p20)
    )
    assert np.isfinite(_orth_bound_p20)
    assert _orth_gap_p20 <= _orth_bound_p20

_G_flow_p20 = _X2_p20.T @ _X2_p20
_c_flow_p20 = _X2_p20.T @ _y2_p20
_sentinel_p20 = np.array([123.25, -77.5])
_flow_calls_p20 = []


def flow_solve_p20(a, b):
    _flow_calls_p20.append((np.array(a, copy=True), np.array(b, copy=True)))
    return _sentinel_p20.copy()


_flow_beta_p20 = _invoke_p20(_X2_p20, _y2_p20, flow_solve_p20)
assert len(_flow_calls_p20) == 1
assert np.array_equal(_flow_calls_p20[0][0], _G_flow_p20)
assert np.array_equal(_flow_calls_p20[0][1], _c_flow_p20)
assert np.array_equal(_flow_beta_p20, _sentinel_p20)

_pre_rejected_p20 = (
    (np.ones(3), np.ones(3)),
    (np.ones((3, 2)), np.ones((3, 1))),
    (np.ones((3, 2)), np.ones(2)),
    (np.ones((2, 3)), np.ones(2)),
    (np.array([[1.0, 2.0], [2.0, 4.0], [3.0, 6.0]]), np.ones(3)),
    (np.array([[1.0, np.nan], [1.0, 2.0]]), np.ones(2)),
)
for _X_bad_p20, _y_bad_p20 in _pre_rejected_p20:
    _solve_calls_p20 = []

    def unexpected_solve_p20(a, b):
        _solve_calls_p20.append((a, b))
        return _ORIGINAL_SOLVE_P20(a, b)

    try:
        _invoke_p20(_X_bad_p20, _y_bad_p20, unexpected_solve_p20)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid input must raise ValueError")
    assert _solve_calls_p20 == []

_X_gram_singular_p20 = np.array([
    [1.0, 1.0],
    [1.0, 1.0 + 1e-9],
    [1.0, 1.0 - 1e-9],
    [1.0, 1.0 + 2e-9],
])
_y_gram_singular_p20 = np.arange(4.0)
assert np.linalg.matrix_rank(_X_gram_singular_p20) == 2
_singular_calls_p20 = []


def singular_solve_p20(a, b):
    _singular_calls_p20.append(
        (np.array(a, copy=True), np.array(b, copy=True))
    )
    return _ORIGINAL_SOLVE_P20(a, b)


try:
    _invoke_p20(
        _X_gram_singular_p20, _y_gram_singular_p20, singular_solve_p20,
    )
except ValueError:
    pass
else:
    raise AssertionError("LinAlgError must be converted to ValueError")
assert len(_singular_calls_p20) == 1
assert np.array_equal(
    _singular_calls_p20[0][0],
    _X_gram_singular_p20.T @ _X_gram_singular_p20,
)
assert np.array_equal(
    _singular_calls_p20[0][1],
    _X_gram_singular_p20.T @ _y_gram_singular_p20,
)

### Answer check

The derivation expands the residual sum of squares coordinate by coordinate, obtains the gradient $\nabla_{\beta}L=(2/n)X^{\mathsf T}(X\beta-y)$ and the normal equations from the first-order condition, proves that full column rank makes $X^{\mathsf T}X$ positive definite and invertible, and connects the unique solution to `G`, `c`, and `np.linalg.solve(G, c)`. The immutable checker audits the direct helper-free bytecode namespace, rejects malformed inputs before solving, instruments exactly one solve call, verifies controlled-return data flow, converts a computed-Gram `LinAlgError` to `ValueError`, and checks full-rank, scaled, ill-conditioned, and rejected fixtures with the prescribed tolerances and backward-error bounds.